# 年度恢复模型时间外概率诊断
固定2bp参考、原模型原概率，不调阈值。训练先验只用各年前标签比例；AUC只比较同年正负样本，不混用不同模型概率。完整日历区块重抽样，缺模型2020保留缺失。

In [1]:
from pathlib import Path
import os,json,hashlib
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr
from src.workflow.minute_action_value import _source
from src.workflow.stage1_open_action_value import Stage1OpenActionValueSpec
from src.workflow.intraday_recovery_direct_allocation import IntradayRecoveryDirectAllocationSpec,FEATURE_COLUMNS,project_legal_weights
from src.analysis.recovery_training_labels import recovery_training_labels
from src.objective.relative_wealth import stationary_block_bootstrap_indices
from src.data.nas_market_data import archive_nas_recorder
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/annual_recovery_information_20261015';p=json.loads((card/'plan.json').read_text());provider=root/'data/qlib/full_path_tiered_allocation_20261013';os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(provider),region='us')
accounts,aa=_source(root,p['account_source'],['inputs/frame','inputs/selection','analysis/native/control_2']);models,ma=_source(root,p['model_source'],['analysis/training_audit','models/2021','models/2022']);action,ac=_source(root,p['action_source'],['inputs/plan'])
frame=accounts['inputs/frame'];book=dict(accounts['analysis/native/control_2']);recovery=IntradayRecoveryDirectAllocationSpec(root=root,**action['inputs/plan']['recovery_spec']);spec=Stage1OpenActionValueSpec(root=root,**action['inputs/plan']['action_spec']);selection=accounts['inputs/selection'];path=book['authority'];assert not path.recovery_action.any() and path.execute_open_target.all()
# Rebuild prescribed targets from saved caps; label helper independently rechecks all states and cash.
weights=[project_legal_weights(float(row.soxl_weight),float(row.soxx_weight),float(path.iloc[i].current_beta_cap),minimum_cash_weight=recovery.minimum_cash_weight,maximum_soxl_weight=recovery.maximum_soxl_weight) for i,row in frame.iterrows()]
book['targets']=pd.DataFrame(weights,columns=['soxl_weight','soxx_weight','cash_weight']);labels=recovery_training_labels(frame,book,plan=p,recovery_spec=recovery,action_spec=spec,selection=selection,slippage_bps=2)
labels['year']=labels.trade_date.dt.year;labels['probability']=np.nan;labels['training_prior']=np.nan;audits=pd.DataFrame(models['analysis/training_audit']).set_index('year')
for year in [2021,2022]:
 a=audits.loc[year];assert a.status=='fitted';mask=labels.year.eq(year)&labels.eligible&labels.net_advantage.notna()&labels.label_available_at.lt(pd.Timestamp('2023-01-01',tz='America/New_York'));assert labels.loc[mask,'decision_at'].gt(pd.Timestamp(a.cutoff)).all();labels.loc[mask,'probability']=models[f'models/{year}'].predict_proba(labels.loc[mask,list(FEATURE_COLUMNS)])[:,1];labels.loc[mask,'training_prior']=a.positive_rows/a.rows
labels['outcome']=labels.net_advantage.gt(0).where(labels.net_advantage.notna()).astype(float)
def stats(data):
 z=data.dropna(subset=['probability','training_prior','outcome','net_advantage']);brier=float(((z.training_prior-z.outcome)**2-(z.probability-z.outcome)**2).mean()) if len(z) else np.nan
 au=[];aw=[];ranks=[];rw=[]
 for _,g in z.groupby('year'):
  pos=int(g.outcome.sum());neg=len(g)-pos
  if pos and neg:au.append(roc_auc_score(g.outcome,g.probability));aw.append(pos*neg)
  if len(g)>2 and g.probability.nunique()>1 and g.net_advantage.nunique()>1:ranks.append(spearmanr(g.probability,g.net_advantage).statistic);rw.append(len(g))
 return [brier,float(np.average(au,weights=aw)) if aw else np.nan,float(np.average(ranks,weights=rw)) if rw else np.nan]
rows=[]
for year,g in [('all',labels),*list(labels.groupby('year'))]:
 z=g.dropna(subset=['probability']);v=stats(g);rows.append({'year':str(year),'scored_actions':len(z),'positive_fraction':float(z.outcome.mean()),'mean_probability':float(z.probability.mean()),'mean_training_prior':float(z.training_prior.mean()),'brier_improvement':v[0],'within_year_pair_weighted_auc':v[1],'within_year_n_weighted_net_spearman':v[2]})
indices=stationary_block_bootstrap_indices(observation_count=len(labels),path_length=len(labels),path_count=p['bootstrap_resamples'],expected_block_length=p['bootstrap_mean_block'],seed=p['bootstrap_seed']);draws=np.asarray([stats(labels.iloc[index]) for index in indices]);point=stats(labels);intervals=[]
for i,(name,null) in enumerate([('brier_improvement',0.),('within_year_auc',.5),('within_year_net_spearman',0.)]):
 v=draws[:,i];v=v[np.isfinite(v)];assert len(v)>=.95*p['bootstrap_resamples'];lo,hi=np.quantile(v,[p['tail_probability'],1-p['tail_probability']]);intervals.append({'metric':name,'point':point[i],'null':null,'lower':lo,'upper':hi,'lower_above_null':bool(lo>null),'valid_resamples':len(v)})
policy={'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/daily_labels','analysis/bootstrap_indices','analysis/bootstrap_statistics']}
with R.start(experiment_name=card.name,uri=str(card/'qlib_recorder')):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=p,source_audits={'accounts':aa,'models':ma,'action':ac});rec.save_objects(artifact_path='analysis',daily_labels=labels,bootstrap_indices=indices,bootstrap_statistics=draws,metrics=pd.DataFrame(rows),intervals=pd.DataFrame(intervals));rec.log_metrics(scored_actions=int(labels.probability.notna().sum()),registered_artifact_count=7)
summary={'experiment':card.name,'decision':'diagnostic_only_no_promotion','new_fits':0,'new_backtests':0,'scored_actions':int(labels.probability.notna().sum()),'positive_information_bounds':sum(i['lower_above_null'] for i in intervals),'provider_uri':str(provider),'region':'us','as_of':p['as_of'],'recorder_id':rec.id,'experiment_id':str(rec.experiment_id),'recorder_uri':str(card/'qlib_recorder'),'recorder_local_dir':str(card/'qlib_recorder'/str(rec.experiment_id)/rec.id),'artifact_policy':policy,'scope':p['scope'],'objective_achieved':False}
pd.DataFrame(rows).to_csv(card/'probability_metrics.csv',index=False);pd.DataFrame(intervals).to_csv(card/'confidence_intervals.csv',index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':str(provider),'artifact_policy':policy}));binding=json.loads((root/p['account_source']['card']/'nas_binding.json').read_text());binding.pop('evidence',None);(card/'nas_binding.json').write_text(json.dumps(binding,indent=2));a=Path(summary['recorder_local_dir'])/'artifacts';(card/'verification_manifest.json').write_text(json.dumps({'recorder_id':rec.id,'entries':[{'path':str(f.relative_to(card)),'sha256':hashlib.sha256(f.read_bytes()).hexdigest()} for f in sorted(a.rglob('*')) if f.is_file()]},indent=2));summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));(card/'summary.json').write_text(json.dumps(summary,indent=2));print(pd.DataFrame(rows).to_string(index=False));print(pd.DataFrame(intervals).to_string(index=False))


# 年度恢复概率诊断：未证明区分力，概率损失差于固定先验

本轮0拟合、0新增完整回测。复用已核验的配对无恢复2bp参考账户与原2021/2022分类器。按保存的风险上限重建目标，并由既有标签函数再次核对逐日状态、股数及现金；以同日恢复/HOLD分支和次日共同重置计算净优势，不修改模型概率、阈值或动作。2020无合格模型，保留缺失而非填0概率。

共265个成熟可评分动作：2021年27、2022年238。对照概率仅用模型训练期正类比例，分别102/174与116/201；未用评价期实际胜率估计对照。AUC只比较同一年度模型的正负样本，再按正负配对数加权；净优势Spearman按年度样本数加权，避免直接混合不同模型概率水平。

| 指标 | 点估计 | 区块端点 | 参考值 |
|---|---:|---:|---:|
| 固定先验Brier减模型Brier | -0.02930 | [-0.05606,-0.00777] | 0 |
| 同年配对加权AUC | 0.51665 | [0.43240,0.60186] | 0.5 |
| 同年净优势秩相关 | 0.01189 | [-0.12068,0.13799] | 0 |

固定均长63日平稳区块重抽样2000次，保留完整756日历和缺失/不合格动作位置。1999次可估，1次因缺可估统计保留NaN，不补零也不追加抽样。三项下界采用单侧Bonferroni校正，端点为1.667%/98.333%分位；不涵盖历史全部研究选择或重新拟合不确定性。

2021实际正类51.85%，平均预测36.18%，年度AUC0.6209，但仅27条；2022实际正类44.12%，平均预测64.54%，年度AUC0.5153。2022有238条，主要体现概率偏高和弱排序；不能只挑2021点估计宣称有效。Brier是概率平方误差，不是策略收益率；其负改善表示模型比事先已知的固定训练胜率更差。

结果不支持把失败仅归为.825阈值过高。即使事后校准可能改变概率水平，也不会凭空创造排序或净优势信息；本轮不拟合校准器、不翻号、不换期或搜索阈值。诊断限定在无恢复参考的局部动作标签，不能把数值全部归为完整候选账户的因果损失，也不证明所有原始分钟变量或其他恢复机制均无效。训练与评价账户状态分布差异仍需保留。

决策diagnostic_only_no_promotion。该固定恢复模型的收益路线已关闭，本轮进一步排除了直接用阈值搜索充作修复的依据。后续需要能够通过时间隔离检验的恢复净优势信息，而非继续在同一复用窗口挑参数。生产未改，稳定净SOXX超额未完成。

执行notebook、逐日概率/标签、区块索引及统计均保存Qlib Recorder，NAS冷恢复逐对象哈希通过。
